In [7]:
#using python3.10 for torch to work properly
#use environment.yml
import glob
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
from datetime import datetime, date
import json
import os
import requests
from pandas import json_normalize
from tqdm import tqdm

# Loading data

In [ ]:
#Import data and forcing format string and dateformat

path = "/Users/pierre-yvesbenevise/Documents/DataEngineer/Projects/citibike_prediction/01_Data/"

#path = "Data/citibike_tripdata"

# List all CSV files in the folder
all_files = glob.glob(path + "nyc_data_raw/citibike_tripdata/*.csv") 

# Columns to keep
columns_to_keep = [
    'started_at', 'ended_at','start_station_id','end_station_id', "rideable_type",
    'member_casual', 'start_lat','start_lng','end_lat','end_lng'
]

dfs = []

# Add csv files 
for file in tqdm(all_files, desc="Loading CSV files"):
    df = pd.read_csv(
        file,
        usecols=columns_to_keep,
        dtype={
            'rideable_type': str,
            'start_station_id': str,
            'end_station_id': str,
            'member_casual': str
        },
        parse_dates=['started_at', 'ended_at'],
        date_format="%Y-%m-%d %H:%M:%S.%f"
    )
    dfs.append(df)
    
biketrip_raw = pd.concat(dfs, ignore_index=True)
biketrip_raw.info()

Loading CSV files: 100%|██████████| 53/53 [00:52<00:00,  1.02it/s]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46283735 entries, 0 to 46283734
Data columns (total 10 columns):
 #   Column            Dtype         
---  ------            -----         
 0   rideable_type     object        
 1   started_at        datetime64[ns]
 2   ended_at          datetime64[ns]
 3   start_station_id  object        
 4   end_station_id    object        
 5   start_lat         float64       
 6   start_lng         float64       
 7   end_lat           float64       
 8   end_lng           float64       
 9   member_casual     object        
dtypes: datetime64[ns](2), float64(4), object(4)
memory usage: 3.4+ GB


In [27]:
#Information stations
url = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json"

# request the data
response = requests.get(url)
response.raise_for_status()

data = response.json()

# convert to dataframe
all_stations_informations = pd.DataFrame(data["data"]["stations"])[
    ["short_name", "station_id", "name", "capacity", "lat", "lon"]
]

In [39]:
#Make sure that the staion IDs in biketrip_raw match the format of short_name in all_stations_informations
all_stations_informations['short_name'] = all_stations_informations['short_name'].astype(str)
biketrip_raw['start_station_id'] = biketrip_raw['start_station_id'].astype(str)
biketrip_raw['end_station_id'] = biketrip_raw['end_station_id'].astype(str)

#short_name values will have two digits after the decimal if they originally had only one
biketrip_raw['start_station_id']= biketrip_raw['start_station_id'].apply(
    lambda s: f"{s.split('.')[0]}.{s.split('.')[1]}0" if '.' in s and len(s.split('.')[1]) == 1 else s
)
biketrip_raw['end_station_id']= biketrip_raw['end_station_id'].apply(
    lambda s: f"{s.split('.')[0]}.{s.split('.')[1]}0" if '.' in s and len(s.split('.')[1]) == 1 else s
)
# Combine all station IDs from trips
trip_station_ids = pd.Series(
    pd.concat([biketrip_raw['start_station_id'], biketrip_raw['end_station_id']])
).unique()

# Identify missing stations
missing_stations = [s for s in trip_station_ids if s not in all_stations_informations['short_name'].values]

# Assuming missing_stations is a list or set of station IDs
biketrip_raw = biketrip_raw[
    ~biketrip_raw['start_station_id'].isin(missing_stations) &
    ~biketrip_raw['end_station_id'].isin(missing_stations)
]
biketrip_raw.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44851809 entries, 0 to 46283734
Data columns (total 10 columns):
 #   Column            Dtype         
---  ------            -----         
 0   rideable_type     object        
 1   started_at        datetime64[ns]
 2   ended_at          datetime64[ns]
 3   start_station_id  object        
 4   end_station_id    object        
 5   start_lat         float64       
 6   start_lng         float64       
 7   end_lat           float64       
 8   end_lng           float64       
 9   member_casual     object        
dtypes: datetime64[ns](2), float64(4), object(4)
memory usage: 3.7+ GB


In [40]:
#Pourcentage de vide
print('Pourcentage de vide biketrip')
print(100*biketrip_raw.isnull().sum()/biketrip_raw.shape[0])

#Display Shape
print('Nombre de lignes biketrip', biketrip_raw.shape[0])
print('Nombre de Colones biketrip', biketrip_raw.shape[1])

#Delete Not full rows -> since a lot of data available
biketrip = biketrip_raw.dropna().copy()

#Converting categorical columns to category type
categorical_cols = [
    "rideable_type",
    "start_station_id",
    "end_station_id",
    "member_casual"
]

for col in categorical_cols:
    biketrip[col] = biketrip[col].astype("category")
    
#Adding date information
biketrip.loc[:, 'started_at_date'] = biketrip['started_at'].dt.floor('D')
biketrip.loc[:, 'started_at_hour'] = biketrip['started_at'].dt.hour
biketrip.loc[:, 'ended_at_date'] = biketrip['ended_at'].dt.floor('D')
biketrip.loc[:, 'ended_at_hour'] = biketrip['ended_at'].dt.hour

# calculating Duration for each trip
biketrip.loc[:, 'duration'] = (
    (biketrip['ended_at'] - biketrip['started_at'])
    .dt.total_seconds()
    .div(60)
    .round(2)
)
#remove data outisde Quantile 1% and 99%, better for mobility dataset 
lower = biketrip['duration'].quantile(0.01)
upper = biketrip['duration'].quantile(0.99)
biketrip = biketrip[(biketrip['duration'] >= lower) &
                    (biketrip['duration'] <= upper)]

#Remove duration < 1minute and >80minutes
biketrip = biketrip[(biketrip['duration'] >= 1) & (biketrip['duration'] <= 80)]

# Vectorized Haversine calculation
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in km

    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    distance = R * c
    return distance

# Apply vectorized function to the DataFrame
biketrip['distance_km'] = haversine_vectorized(
    biketrip['start_lat'].values,
    biketrip['start_lng'].values,
    biketrip['end_lat'].values,
    biketrip['end_lng'].values
)

# Round distances to 2 decimals
biketrip['distance_km'] = biketrip['distance_km'].round(2)

#remove data outisde +/- 3sigma
mu = biketrip['distance_km'].mean()
sigma = biketrip['distance_km'].std()

lower_bound = mu - 3 * sigma
upper_bound = mu + 3 * sigma

# Filter rows inside the range
biketrip = biketrip[(biketrip['distance_km'] >= lower_bound) & 
                          (biketrip['distance_km'] <= upper_bound)]
biketrip.info()

Pourcentage de vide biketrip
rideable_type       0.0
started_at          0.0
ended_at            0.0
start_station_id    0.0
end_station_id      0.0
start_lat           0.0
start_lng           0.0
end_lat             0.0
end_lng             0.0
member_casual       0.0
dtype: float64
Nombre de lignes biketrip 44851809
Nombre de Colones biketrip 10
<class 'pandas.core.frame.DataFrame'>
Index: 43127128 entries, 0 to 46283734
Data columns (total 16 columns):
 #   Column            Dtype         
---  ------            -----         
 0   rideable_type     category      
 1   started_at        datetime64[ns]
 2   ended_at          datetime64[ns]
 3   start_station_id  category      
 4   end_station_id    category      
 5   start_lat         float64       
 6   start_lng         float64       
 7   end_lat           float64       
 8   end_lng           float64       
 9   member_casual     category      
 10  started_at_date   datetime64[ns]
 11  started_at_hour   int32         
 12  ende

In [41]:
#For All stations
biketrip_for_modeling_all_stations = biketrip.drop(['start_lat','start_lng','end_lat','end_lng','started_at','ended_at',
                                       'duration','distance_km',"rideable_type",'member_casual'], axis = 1)

# Bikes taken (pickups) for the selected station
hourly_pickups = biketrip_for_modeling_all_stations.groupby(
    ['start_station_id','started_at_date','started_at_hour'],
    observed=True
).size().reset_index(name='num_bikes_taken')
hourly_pickups.rename(columns={'started_at_date': 'date', 'started_at_hour': 'hour','start_station_id':'station_id'}, inplace=True)

# Bikes dropped (returns) for the selected station
hourly_drops = biketrip_for_modeling_all_stations.groupby(
    ['end_station_id','ended_at_date','ended_at_hour'],
    observed=True
).size().reset_index(name='num_bikes_dropped')
hourly_drops.rename(columns={'ended_at_date': 'date', 'ended_at_hour': 'hour','end_station_id':'station_id'}, inplace=True)

# Merge pickups and drops into a single dataframe
hourly_counts_all_stations = pd.merge(hourly_pickups, hourly_drops, on=['station_id', 'date', 'hour'], how='outer').fillna(0)

# Sanity check
hourly_counts_all_stations["date_hour"] = pd.to_datetime(hourly_counts_all_stations["date"]) + pd.to_timedelta(hourly_counts_all_stations["hour"], unit="h")
dups = hourly_counts_all_stations[hourly_counts_all_stations.duplicated(subset=["station_id", "date_hour"], keep=False)]
print('Merge pickups and drops into a single dataframe duplicate', dups.shape[0])

# Force numeric type
hourly_counts_all_stations['num_bikes_taken'] = hourly_counts_all_stations['num_bikes_taken'].astype(int)
hourly_counts_all_stations['num_bikes_dropped'] = hourly_counts_all_stations['num_bikes_dropped'].astype(int)

# Add flow
# sort by day and hour 
hourly_counts_all_stations = hourly_counts_all_stations.sort_values(['station_id','date', 'hour']).reset_index(drop=True)

hourly_counts_all_stations['net_flow'] = (hourly_counts_all_stations['num_bikes_dropped'] - hourly_counts_all_stations['num_bikes_taken']).astype(int)

# ensure date is datetime
hourly_counts_all_stations['date'] = pd.to_datetime(hourly_counts_all_stations['date'])

# create full index: station × date × hour
full_index = pd.MultiIndex.from_product(
    [
        hourly_counts_all_stations['station_id'].unique(),
        pd.date_range(hourly_counts_all_stations['date'].min(), hourly_counts_all_stations['date'].max(), freq='D'),
        range(24)
    ],
    names=['station_id', 'date', 'hour']
)

# set index on existing data
hourly_counts_all_stations_with_net_flow = (
    hourly_counts_all_stations
    .set_index(['station_id', 'date', 'hour'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

hourly_counts_all_stations_with_net_flow['station_id'] = (
    hourly_counts_all_stations_with_net_flow['station_id'].astype('category')
)

# Sanity check
hourly_counts_all_stations_with_net_flow["date_hour"] = pd.to_datetime(hourly_counts_all_stations_with_net_flow["date"]) + pd.to_timedelta(hourly_counts_all_stations_with_net_flow["hour"], unit="h")
dups = hourly_counts_all_stations_with_net_flow[hourly_counts_all_stations_with_net_flow.duplicated(subset=["station_id", "date_hour"], keep=False)]
print('Couple sation_id et date en double', dups.shape[0])

hourly_counts_all_stations_with_net_flow.info()

Merge pickups and drops into a single dataframe duplicate 0
Couple sation_id et date en double 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19728864 entries, 0 to 19728863
Data columns (total 7 columns):
 #   Column             Dtype         
---  ------             -----         
 0   station_id         category      
 1   date               datetime64[ns]
 2   hour               int64         
 3   num_bikes_taken    int64         
 4   num_bikes_dropped  int64         
 5   date_hour          datetime64[ns]
 6   net_flow           int64         
dtypes: category(1), datetime64[ns](2), int64(4)
memory usage: 940.8 MB


In [ ]:
#load weather into dataframe from computer
weather_df = pd.read_csv('/Users/pierre-yvesbenevise/Documents/DataEngineer/Projects/citibike_prediction/01_Data/Eda/nyc_data_raw/weather_data_2024_11_01_2025_12_01.csv')
weather_df = weather_df.astype({
    "date": "datetime64[ns]",
    'hour': 'int64',
    "temp": "float32",
    "relative_humidity": "float32",
    "precipitation_total": "float32",
    "average_wind_speed": "float32",
    "coco": "category"
})
start_date = weather_df['date'].min()
end_date = weather_df['date'].max()

hourly_counts_all_stations_with_net_flow = (
    hourly_counts_all_stations_with_net_flow[
        (hourly_counts_all_stations_with_net_flow['date'] >= start_date) &
        (hourly_counts_all_stations_with_net_flow['date'] <= end_date)
    ]
)

# Merge biketrip with weather on the date
hourly_counts_all_stations_with_net_flow_weather= hourly_counts_all_stations_with_net_flow.merge(weather_df,
    on=['date', 'hour'],
    how='left'
)

# Sanity check
hourly_counts_all_stations_with_net_flow_weather["date_hour"] = pd.to_datetime(hourly_counts_all_stations_with_net_flow_weather["date"]) + pd.to_timedelta(hourly_counts_all_stations_with_net_flow_weather["hour"], unit="h")
dups = hourly_counts_all_stations_with_net_flow_weather[hourly_counts_all_stations_with_net_flow_weather.duplicated(subset=["station_id", "date_hour"], keep=False)]
print('Merge biketrip with weather on the date duplicate', dups.shape[0])

hourly_counts_all_stations_with_net_flow_weather['is_rain'] = (hourly_counts_all_stations_with_net_flow_weather['precipitation_total'] > 0).astype(int)

weather_df.info()
hourly_counts_all_stations_with_net_flow_weather.describe(include = 'all').T
print(hourly_counts_all_stations_with_net_flow_weather.head())

Merge biketrip with weather on the date duplicate 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9504 entries, 0 to 9503
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 9504 non-null   datetime64[ns]
 1   hour                 9504 non-null   int64         
 2   temp                 9504 non-null   float32       
 3   relative_humidity    9504 non-null   float32       
 4   precipitation_total  9504 non-null   float32       
 5   average_wind_speed   9504 non-null   float32       
 6   coco                 9504 non-null   category      
dtypes: category(1), datetime64[ns](1), float32(4), int64(1)
memory usage: 307.1 KB
  station_id       date  hour  num_bikes_taken  num_bikes_dropped  \
0    2261.04 2024-11-01     0                0                  0   
1    2261.04 2024-11-01     1                0                  0   
2    2261.04 2024-11-01     2         

In [ ]:
#load holidays into dataframe from Computer
holidays_df = pd.read_csv( '/Users/pierre-yvesbenevise/Documents/DataEngineer/Projects/citibike_prediction/01_Data/raw/holidays.csv')
holidays_df = holidays_df.astype({
    "date": "datetime64[ns]",
    'is_holiday': 'bool'
})
holidays_df.info()

holiday_dates = holidays_df['date'].dt.normalize()

hourly_counts_all_stations_with_net_flow_weather_holidays = hourly_counts_all_stations_with_net_flow_weather.copy()

hourly_counts_all_stations_with_net_flow_weather_holidays['is_holiday'] = (
    hourly_counts_all_stations_with_net_flow_weather_holidays['date']
    .dt.normalize()
    .isin(holiday_dates)
)

# Sanity check
hourly_counts_all_stations_with_net_flow_weather_holidays["date_hour"] = pd.to_datetime(hourly_counts_all_stations_with_net_flow_weather_holidays["date"]) + pd.to_timedelta(hourly_counts_all_stations_with_net_flow_weather_holidays["hour"], unit="h")
dups = hourly_counts_all_stations_with_net_flow_weather_holidays[hourly_counts_all_stations_with_net_flow_weather_holidays.duplicated(subset=["station_id", "date_hour"], keep=False)]
print('Merge biketrip with holidays on the date duplicate', dups.shape[0])

hourly_counts_all_stations_with_net_flow_weather_holidays.describe(include = 'all').T
print(hourly_counts_all_stations_with_net_flow_weather_holidays.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        13 non-null     datetime64[ns]
 1   is_holiday  13 non-null     bool          
dtypes: bool(1), datetime64[ns](1)
memory usage: 245.0 bytes
Merge biketrip with holidays on the date duplicate 0
  station_id       date  hour  num_bikes_taken  num_bikes_dropped  \
0    2261.04 2024-11-01     0                0                  0   
1    2261.04 2024-11-01     1                0                  0   
2    2261.04 2024-11-01     2                0                  0   
3    2261.04 2024-11-01     3                0                  0   
4    2261.04 2024-11-01     4                0                  0   

            date_hour  net_flow  temp  relative_humidity  precipitation_total  \
0 2024-11-01 00:00:00         0  23.0               52.0                  0.0   
1 2024-11-01 01

In [44]:
hourly_counts_all_stations_complete = hourly_counts_all_stations_with_net_flow_weather_holidays.copy()

hourly_counts_all_stations_complete['date'] = pd.to_datetime(hourly_counts_all_stations_complete['date'], errors='coerce')
hourly_counts_all_stations_complete['year'] =hourly_counts_all_stations_complete['date'].dt.year
hourly_counts_all_stations_complete['month'] =hourly_counts_all_stations_complete['date'].dt.month
hourly_counts_all_stations_complete['day'] =hourly_counts_all_stations_complete['date'].dt.day
hourly_counts_all_stations_complete['jour_semaine'] =hourly_counts_all_stations_complete['date'].dt.day_name(locale='fr_FR').astype('category')

hourly_counts_all_stations_complete = hourly_counts_all_stations_complete[['station_id','date', 'date_hour', 'year', 'month', 'day', 'jour_semaine', 'hour', 'num_bikes_taken', 'num_bikes_dropped', 'net_flow',
                                                      'temp', 'relative_humidity', 'precipitation_total', 'average_wind_speed',
                                                      'coco', 'is_holiday']]

#Pourcentage de vide
print('Pourcentage de vide hourly_counts_all_stations')
print(100*hourly_counts_all_stations_complete.isnull().sum()/hourly_counts_all_stations_complete.shape[0])

#Display Shape
print('Nombre de lignes hourly_counts_all_stations', hourly_counts_all_stations_complete.shape[0])
print('Nombre de Colones hourly_counts_all_stations', hourly_counts_all_stations_complete.shape[1])

hourly_counts_all_stations_complete.info()

hourly_counts_all_stations_complete.describe(include = 'all').T

print(hourly_counts_all_stations_complete.head())

Pourcentage de vide hourly_counts_all_stations
station_id             0.0
date                   0.0
date_hour              0.0
year                   0.0
month                  0.0
day                    0.0
jour_semaine           0.0
hour                   0.0
num_bikes_taken        0.0
num_bikes_dropped      0.0
net_flow               0.0
temp                   0.0
relative_humidity      0.0
precipitation_total    0.0
average_wind_speed     0.0
coco                   0.0
is_holiday             0.0
dtype: float64
Nombre de lignes hourly_counts_all_stations 19674960
Nombre de Colones hourly_counts_all_stations 17
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19674960 entries, 0 to 19674959
Data columns (total 17 columns):
 #   Column               Dtype         
---  ------               -----         
 0   station_id           category      
 1   date                 datetime64[ns]
 2   date_hour            datetime64[ns]
 3   year                 int32         
 4   month       

In [ ]:
filename_final = '/Users/pierre-yvesbenevise/Documents/DataEngineer/Projects/citibike_prediction/01_Data/hourly_counts_all_stations.parquet'
if filename_final in os.listdir():
        os.remove(filename_final)
hourly_counts_all_stations_complete.to_parquet(filename_final, index=False)